# Notebook 11 — Prefix Caching, Chunked Prefill, and Long-Context Runtime Behavior

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/systems/11_prefix_caching_chunked_prefill_and_long_context.ipynb)

Serving performance does not come only from faster kernels. It also comes from giving the runtime prompts and scheduling conditions it can exploit. This notebook focuses on prefix reuse, chunked prefill, and the operational reality of long-context inference.

## What you will build

- a workload with stable shared prefixes and variable suffixes
- a prefix-cache simulator that estimates prefill savings
- a monolithic versus chunked-prefill comparison
- a long-context memory and latency model
- a cache-aware prompt design brief for later modules

## Design rule

If you want the runtime to help you, keep the reusable part of the prompt stable and the changing part late. Prefix caching only works when repeated prompt structure is actually repeated.

In [ ]:
# --- Systems Notebook Setup ---
!pip install -q "transformers>=4.51.0" accelerate torch numpy pandas matplotlib fastapi uvicorn pydantic httpx psutil

import asyncio
import json
import math
import random
import statistics
import threading
import time
from collections import Counter, defaultdict, deque
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 140)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = Path("/content/drive/MyDrive/models")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/castalia-systems")
except Exception:
    CACHE_DIR = Path("./models")
    ARTIFACT_ROOT = Path("./artifacts")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

def now_ms():
    return time.perf_counter() * 1000

def summarize(values):
    return {
        "count": len(values),
        "mean": statistics.mean(values) if values else 0.0,
        "median": statistics.median(values) if values else 0.0,
        "min": min(values) if values else 0.0,
        "max": max(values) if values else 0.0,
    }

print("✓ Systems notebook environment ready")
print("  Cache dir:", CACHE_DIR)
print("  Artifact root:", ARTIFACT_ROOT)

## Step 1: Add helpers and an artifact path

We will export prefix-cache summaries, chunked-prefill comparisons, and long-context operating notes.

In [ ]:
random.seed(11)

ARTIFACT_DIR = ARTIFACT_ROOT / "11_prefix_caching_chunked_prefill_and_long_context"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def write_json(path, payload):
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")

def approx_token_count(text):
    return max(1, int(len(text.split()) * 1.35))

def percentile(values, pct):
    ordered = sorted(float(v) for v in values)
    if not ordered:
        return 0.0
    rank = (len(ordered) - 1) * (pct / 100)
    lower = math.floor(rank)
    upper = math.ceil(rank)
    if lower == upper:
        return ordered[int(rank)]
    weight = rank - lower
    return ordered[lower] * (1 - weight) + ordered[upper] * weight

print("Artifact directory:", ARTIFACT_DIR.resolve())

## Step 2: Define reusable prompt prefixes

These prefixes represent the stable front half of prompts that often appears in assistant, RAG, and agent-serving workloads.

In [ ]:
shared_prefixes = {
    "mentor-chat": "System: You are Castalia Mentor. Follow the course style guide. Cite runtime behavior plainly. Keep explanations concrete and operational.",
    "rag-answer": "System: Answer using the provided retrieved notes. Preserve section boundaries. Prefer direct evidence and highlight uncertainty when evidence is thin.",
    "tool-audit": "System: Review a tool trace for correctness. Explain the action sequence, note contract violations, and return a concise remediation plan.",
}

prefix_df = pd.DataFrame([{
    "prefix_template": name,
    "prefix_tokens_est": approx_token_count(text),
    "preview": text[:110] + "...",
} for name, text in shared_prefixes.items()])
print(prefix_df.to_markdown(index=False))

## Step 3: Create a workload that reuses those prefixes

The suffixes vary by user question and retrieved evidence, but the leading system and policy text stays stable enough to cache.

In [ ]:
workload_rows = [
    {"request_id": "pc01", "arrival_ms": 0, "prefix_template": "mentor-chat", "unique_suffix": "User: Explain when continuous batching helps more than larger static batches.", "output_tokens": 160},
    {"request_id": "pc02", "arrival_ms": 7, "prefix_template": "mentor-chat", "unique_suffix": "User: Contrast vLLM and a plain transformers loop for throughput planning.", "output_tokens": 170},
    {"request_id": "pc03", "arrival_ms": 13, "prefix_template": "rag-answer", "unique_suffix": "Context chunk A: Paged KV caches use reusable blocks. Context chunk B: Prefix caching reduces repeated prefill work. Question: connect the two ideas.", "output_tokens": 180},
    {"request_id": "pc04", "arrival_ms": 21, "prefix_template": "rag-answer", "unique_suffix": "Context chunk A: Chunked prefill prevents long prompts from monopolizing the scheduler. Context chunk B: Long context stresses memory. Question: explain the trade-off.", "output_tokens": 200},
    {"request_id": "pc05", "arrival_ms": 29, "prefix_template": "tool-audit", "unique_suffix": "Tool trace: router chose long-context model, parser validated JSON, retry loop fired twice. Diagnose the likely runtime issue.", "output_tokens": 150},
    {"request_id": "pc06", "arrival_ms": 36, "prefix_template": "mentor-chat", "unique_suffix": "User: Give three cache-aware prompt patterns for a tutoring assistant.", "output_tokens": 120},
    {"request_id": "pc07", "arrival_ms": 42, "prefix_template": "tool-audit", "unique_suffix": "Tool trace: prefix cache hits dropped after prompt builder changed system headers. Explain what probably broke.", "output_tokens": 140},
    {"request_id": "pc08", "arrival_ms": 49, "prefix_template": "rag-answer", "unique_suffix": "Context chunk A: Retrieval added six documents and a safety policy appendix. Question: why did TTFT jump?", "output_tokens": 150},
]

workload_df = pd.DataFrame(workload_rows)
workload_df["prefix_tokens"] = workload_df["prefix_template"].map(lambda name: approx_token_count(shared_prefixes[name]))
workload_df["suffix_tokens"] = workload_df["unique_suffix"].map(approx_token_count)
workload_df["prompt_tokens"] = workload_df["prefix_tokens"] + workload_df["suffix_tokens"]
print(workload_df[["request_id", "prefix_template", "arrival_ms", "prefix_tokens", "suffix_tokens", "prompt_tokens", "output_tokens"]].to_markdown(index=False))

## Step 4: Estimate the reusable portion of each prompt

This is the part a prefix cache can hit. If the stable prefix is small, the gain will be modest. If it dominates the prompt, the gain can be dramatic.

In [ ]:
reuse_summary = (
    workload_df.groupby("prefix_template")
    .agg(
        requests=("request_id", "count"),
        avg_prefix_tokens=("prefix_tokens", "mean"),
        avg_prompt_tokens=("prompt_tokens", "mean"),
    )
    .reset_index()
)
reuse_summary["reusable_share"] = (reuse_summary["avg_prefix_tokens"] / reuse_summary["avg_prompt_tokens"]).round(3)
print(reuse_summary.to_markdown(index=False))

## Step 5: Simulate prefix caching economics

The first request for a template pays full prefill. Later requests can reuse the cached prefix and only prefill the changing suffix.

In [ ]:
def simulate_prefix_cache(records, prefill_tps=9000, cache_lookup_ms=4):
    cached_templates = set()
    rows = []
    for record in sorted(records, key=lambda item: item["arrival_ms"]):
        cache_hit = record["prefix_template"] in cached_templates
        cached_tokens = record["prefix_tokens"] if cache_hit else 0
        uncached_tokens = record["prompt_tokens"] - cached_tokens
        full_prefill_ms = 1000 * record["prompt_tokens"] / prefill_tps
        cached_prefill_ms = 1000 * uncached_tokens / prefill_tps + cache_lookup_ms
        rows.append({
            **record,
            "cache_hit": cache_hit,
            "cached_tokens": cached_tokens,
            "full_prefill_ms": round(full_prefill_ms, 1),
            "cached_prefill_ms": round(cached_prefill_ms, 1),
            "prefill_saved_ms": round(full_prefill_ms - cached_prefill_ms, 1),
        })
        cached_templates.add(record["prefix_template"])
    return pd.DataFrame(rows)

prefix_cache_df = simulate_prefix_cache(workload_df.to_dict("records"))
print(prefix_cache_df[["request_id", "prefix_template", "cache_hit", "cached_tokens", "full_prefill_ms", "cached_prefill_ms", "prefill_saved_ms"]].to_markdown(index=False))

## Step 6: Summarize prefix-cache savings

This is the practical reason teams care about prompt stability: the savings show up in prefill latency and system capacity.

In [ ]:
prefix_summary = pd.DataFrame([{
    "requests": int(len(prefix_cache_df)),
    "cache_hits": int(prefix_cache_df["cache_hit"].sum()),
    "hit_rate": round(float(prefix_cache_df["cache_hit"].mean()), 3),
    "total_prefill_saved_ms": round(float(prefix_cache_df["prefill_saved_ms"].sum()), 1),
    "mean_prefill_saved_ms": round(float(prefix_cache_df["prefill_saved_ms"].mean()), 1),
}])
print(prefix_summary.to_markdown(index=False))

## Step 7: Visualize cache savings by request

This makes it easy to see which prompt families are worth optimizing for prefix reuse.

In [ ]:
ax = prefix_cache_df.plot(x="request_id", y=["full_prefill_ms", "cached_prefill_ms"], kind="bar", rot=0, figsize=(9, 4), title="Prefill time before and after prefix caching")
ax.set_ylabel("milliseconds")
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "11_prefix_cache_savings.png", dpi=160)
plt.show()

## Step 8: Write prompt-design rules for cache hits

The cache is only helpful if application code preserves stable leading text.

In [ ]:
prompt_rules = pd.DataFrame([
    {"pattern": "Keep stable system headers first", "why_it_matters": "Prefix caching only helps on the repeated leading span."},
    {"pattern": "Append user-specific data late", "why_it_matters": "Late-changing content preserves more reusable prefix tokens."},
    {"pattern": "Avoid timestamp noise in the prompt prefix", "why_it_matters": "Small dynamic fields can invalidate otherwise reusable cache entries."},
    {"pattern": "Normalize retrieval wrappers", "why_it_matters": "Consistent formatting keeps semantically identical prefixes byte-stable."},
])
print(prompt_rules.to_markdown(index=False))

## Step 9: Define a long-context workload

Now we switch from cache reuse to the harder problem: very large prompts that threaten TTFT and memory budgets.

In [ ]:
long_context_rows = [
    {"request_id": "lc01", "arrival_ms": 0, "context_tokens": 8192, "output_tokens": 220, "workload": "chat+docs"},
    {"request_id": "lc02", "arrival_ms": 10, "context_tokens": 32768, "output_tokens": 260, "workload": "rag synthesis"},
    {"request_id": "lc03", "arrival_ms": 16, "context_tokens": 65536, "output_tokens": 280, "workload": "audit replay"},
    {"request_id": "lc04", "arrival_ms": 24, "context_tokens": 131072, "output_tokens": 320, "workload": "tool history"},
    {"request_id": "lc05", "arrival_ms": 31, "context_tokens": 196608, "output_tokens": 340, "workload": "multi-doc review"},
]

long_context_df = pd.DataFrame(long_context_rows)
print(long_context_df.to_markdown(index=False))

## Step 10: Simulate monolithic prefill

A monolithic scheduler lets each long prompt occupy the front of the line until its entire prefill finishes.

In [ ]:
def simulate_monolithic_prefill(records, prefill_tps=8200, decode_tps=72, overhead_ms=18):
    rows = []
    clock_ms = 0.0
    for record in records:
        queue_ms = max(0.0, clock_ms - record["arrival_ms"])
        prefill_ms = 1000 * record["context_tokens"] / prefill_tps
        decode_ms = 1000 * record["output_tokens"] / decode_tps
        ttft_ms = queue_ms + prefill_ms + 12
        latency_ms = queue_ms + prefill_ms + decode_ms + overhead_ms
        finish_ms = record["arrival_ms"] + latency_ms
        clock_ms = finish_ms
        rows.append({
            **record,
            "policy": "monolithic_prefill",
            "ttft_ms": round(ttft_ms, 1),
            "latency_ms": round(latency_ms, 1),
        })
    return pd.DataFrame(rows)

monolithic_df = simulate_monolithic_prefill(long_context_df.to_dict("records"))
print(monolithic_df.to_markdown(index=False))

## Step 11: Simulate chunked prefill

Chunked prefill breaks large prompts into smaller slices so other requests can make progress between chunks.

In [ ]:
def simulate_chunked_prefill(records, chunk_tokens=4096, prefill_tps=8200, decode_tps=72, overhead_ms=6):
    waiting = deque(sorted([{**record} for record in records], key=lambda item: item["arrival_ms"]))
    active = []
    completed = []
    time_ms = 0.0

    while waiting or active:
        if not active and waiting and waiting[0]["arrival_ms"] > time_ms:
            time_ms = float(waiting[0]["arrival_ms"])
        while waiting and waiting[0]["arrival_ms"] <= time_ms:
            request = waiting.popleft()
            request["remaining_prefill"] = request["context_tokens"]
            request["prefill_chunks"] = 0
            active.append(request)

        next_active = []
        for request in active:
            chunk = min(chunk_tokens, request["remaining_prefill"])
            time_ms += 1000 * chunk / prefill_tps + overhead_ms
            request["remaining_prefill"] -= chunk
            request["prefill_chunks"] += 1
            if request["remaining_prefill"] <= 0:
                ttft_ms = time_ms - request["arrival_ms"] + 12
                decode_ms = 1000 * request["output_tokens"] / decode_tps
                completed.append({
                    "request_id": request["request_id"],
                    "arrival_ms": request["arrival_ms"],
                    "context_tokens": request["context_tokens"],
                    "output_tokens": request["output_tokens"],
                    "workload": request["workload"],
                    "policy": "chunked_prefill",
                    "prefill_chunks": request["prefill_chunks"],
                    "ttft_ms": round(ttft_ms, 1),
                    "latency_ms": round(ttft_ms + decode_ms, 1),
                })
            else:
                next_active.append(request)
        active = next_active

    return pd.DataFrame(completed)

chunked_df = simulate_chunked_prefill(long_context_df.to_dict("records"))
print(chunked_df.to_markdown(index=False))

## Step 12: Compare monolithic and chunked prefill

Chunking does not make the model free. It mainly prevents one very long prefill from monopolizing the scheduler.

In [ ]:
prefill_compare = monolithic_df.merge(chunked_df[["request_id", "prefill_chunks", "ttft_ms", "latency_ms"]], on="request_id", suffixes=("_mono", "_chunked"))
prefill_compare["ttft_saved_ms"] = (prefill_compare["ttft_ms_mono"] - prefill_compare["ttft_ms_chunked"]).round(1)
prefill_compare["latency_saved_ms"] = (prefill_compare["latency_ms_mono"] - prefill_compare["latency_ms_chunked"]).round(1)
print(prefill_compare.to_markdown(index=False))

## Step 13: Plot TTFT under chunked prefill

This chart highlights the user-facing part of the story: shorter time to the first visible token.

In [ ]:
plot_df = prefill_compare[["request_id", "ttft_ms_mono", "ttft_ms_chunked"]].set_index("request_id")
ax = plot_df.plot(kind="bar", rot=0, figsize=(9, 4), title="TTFT before and after chunked prefill")
ax.set_ylabel("milliseconds")
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "11_chunked_prefill_ttft.png", dpi=160)
plt.show()

## Step 14: Estimate KV-cache pressure for long contexts

Long-context serving is often memory-limited before it is purely compute-limited.

In [ ]:
def estimate_kv_gib(context_tokens, concurrent_sequences, hidden_size=8192, layers=80, bytes_per_value=2):
    kv_bytes = concurrent_sequences * context_tokens * hidden_size * layers * 2 * bytes_per_value
    return round(kv_bytes / (1024 ** 3), 1)

memory_rows = []
for context_tokens in [8192, 32768, 65536, 131072, 262144]:
    for concurrent_sequences in [1, 4, 8]:
        memory_rows.append({
            "context_tokens": context_tokens,
            "concurrent_sequences": concurrent_sequences,
            "kv_gib_est": estimate_kv_gib(context_tokens, concurrent_sequences),
        })

memory_df = pd.DataFrame(memory_rows)
print(memory_df.to_markdown(index=False))

## Step 15: Visualize long-context memory growth

This is why long-context deployments quickly become concurrency-planning exercises.

In [ ]:
pivot_df = memory_df.pivot(index="context_tokens", columns="concurrent_sequences", values="kv_gib_est")
ax = pivot_df.plot(marker="o", figsize=(8, 4), title="Estimated KV-cache footprint by context and concurrency")
ax.set_xlabel("context tokens")
ax.set_ylabel("estimated KV cache GiB")
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "11_long_context_memory.png", dpi=160)
plt.show()

## Step 16: Model runtime behavior as context grows

Longer prompts usually push TTFT up and delivered decode rate down, even before you hit an out-of-memory boundary.

In [ ]:
behavior_rows = []
for context_tokens in [8192, 32768, 65536, 131072, 262144]:
    scaling_factor = 1 + math.log2(context_tokens / 8192) * 0.22
    ttft_ms = round(1000 * context_tokens / 9200 * scaling_factor + 14, 1)
    decode_tps = round(82 / scaling_factor, 1)
    behavior_rows.append({
        "context_tokens": context_tokens,
        "scaling_factor": round(scaling_factor, 3),
        "ttft_ms_est": ttft_ms,
        "decode_tps_est": decode_tps,
    })

behavior_df = pd.DataFrame(behavior_rows)
print(behavior_df.to_markdown(index=False))

## Step 17: Plot long-context slowdown

The trend line is what matters: bigger context windows usually buy flexibility at the cost of TTFT and concurrency.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
behavior_df.plot(x="context_tokens", y="ttft_ms_est", marker="o", ax=axes[0], title="TTFT estimate")
behavior_df.plot(x="context_tokens", y="decode_tps_est", marker="o", ax=axes[1], title="Decode rate estimate", color="#f58518")
axes[0].set_ylabel("milliseconds")
axes[1].set_ylabel("tokens per second")
for ax in axes:
    ax.set_xlabel("context tokens")
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "11_long_context_behavior.png", dpi=160)
plt.show()

## Step 18: Turn behavior into admission heuristics

A serving system needs policy, not only theory. These rules are intentionally simple and operational.

In [ ]:
def recommend_lane(context_tokens, prefix_hit_rate, active_long_context):
    if context_tokens >= 131072 and active_long_context >= 3:
        return "defer-or-route-to-long-context-pool"
    if prefix_hit_rate >= 0.6 and context_tokens <= 65536:
        return "prefer-shared-prefix-lane"
    if context_tokens >= 65536:
        return "enable-chunked-prefill"
    return "default-chat-lane"

policy_examples = [
    {"context_tokens": 32000, "prefix_hit_rate": 0.75, "active_long_context": 1},
    {"context_tokens": 70000, "prefix_hit_rate": 0.25, "active_long_context": 1},
    {"context_tokens": 160000, "prefix_hit_rate": 0.10, "active_long_context": 4},
]

policy_df = pd.DataFrame(policy_examples)
policy_df["recommended_lane"] = policy_df.apply(lambda row: recommend_lane(int(row["context_tokens"]), float(row["prefix_hit_rate"]), int(row["active_long_context"])), axis=1)
print(policy_df.to_markdown(index=False))

## Step 19: Export a cache-aware serving brief

This brief captures the main lessons so later notebooks can build on them without repeating the analysis.

In [ ]:
serving_brief = {
    "notebook": "11_prefix_caching_chunked_prefill_and_long_context",
    "prefix_summary": prefix_summary.to_dict("records"),
    "prefill_compare": prefill_compare.to_dict("records"),
    "memory_summary": memory_df.to_dict("records"),
    "behavior_summary": behavior_df.to_dict("records"),
    "prompt_rules": prompt_rules.to_dict("records"),
    "policy_examples": policy_df.to_dict("records"),
    "artifacts": [
        "11_prefix_cache_savings.png",
        "11_chunked_prefill_ttft.png",
        "11_long_context_memory.png",
        "11_long_context_behavior.png",
    ],
}

write_json(ARTIFACT_DIR / "serving_brief.json", serving_brief)
print(json.dumps(serving_brief, indent=2))

## Recap

You now have the prompt-side and scheduler-side intuition for modern long-context serving: preserve stable prefixes, reuse them aggressively, chunk large prefills so the queue keeps moving, and treat long context as a memory-and-admission problem instead of only a larger number on a model card.